# EGD Phase 2 — fast/slow × m=0/m=0.9 (4 configs × 5 seeds × 2 phases)

**Phase 1** : train (sans Fourier) → curves → identification key_freqs par seed (post-hoc sur model.pt final).

**Phase 2** : re-train avec `fixed_key_freqs` per-seed → Fourier loss + concentration sur les freqs identifiées.

`adaptive_logging=True` : eval/fourier_every=10 jusqu'au grok, puis ×5 (→ 50).

`num_epochs=25_000` partagé pour comparabilité avec Muon/AdamW.

## Setup

In [1]:
#!pip install -q torch plotly einops

In [2]:
from pathlib import Path
import numpy as np
import torch as t
import matplotlib.pyplot as plt
import importlib, pipeline, model, viz_analysis
importlib.reload(pipeline); importlib.reload(model); importlib.reload(viz_analysis)

from model        import Config
from pipeline     import OptimizerSpec, Trainer, run_multi_seed
from viz_analysis import (
    load_seeds, compute_grokking_markers, per_seed_stats,
    plot_curves_band, plot_fourier_losses_band,
    plot_freq_mass_heatmap, plot_sparsity,
    plot_fourier_components, plot_fourier_components_per_seed, identify_key_freqs,
)

%matplotlib inline
plt.rcParams.update({'figure.facecolor':'white','figure.dpi':100,'savefig.dpi':200,'font.size':11})
t.manual_seed(0); np.random.seed(0)

ROOT      = Path.cwd()
SAVE_ROOT = ROOT / 'runs' / 'phase2_optim'
FIG_DIR   = ROOT / 'figs_phase2_optim'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# ── Common training settings (partagés Nanda + Muon + AdamW) ────────────────
SEEDS            = [0, 1, 2, 3, 4]
NUM_EPOCHS       = 25_000
EVAL_EVERY       = 10
FOURIER_EVERY    = 10
WARMUP_STEPS     = 10
CONVERGED_THRESH = 0.99

BASE_CONFIG = Config(
    p=113, d_model=128, d_mlp=512, num_heads=4, n_ctx=3,
    act_type='ReLU', frac_train=0.3,
    num_epochs=NUM_EPOCHS, seed=0,
    adaptive_logging=True, adaptive_logging_thresh=0.99, adaptive_logging_factor=5,
)

FILTER_INTERNAL_2D = lambda n, p: p.ndim >= 2 and 'embed' not in n

def make_egd_hybrid(lr_egd, wd_egd, momentum_egd):
    return [
        OptimizerSpec(name='egd', lr=lr_egd, weight_decay=wd_egd,
                      param_filter=FILTER_INTERNAL_2D,
                      extra={'momentum': momentum_egd, 'mode': 'svd'}),
        OptimizerSpec(name='adamw', lr=1e-3, weight_decay=1.0, extra={'betas': (0.9, 0.98)}),
    ]

# ── CONFIGS table : 4 configs EGD ───────────────────────────────────────────
CONFIGS = {
    'egd_m0_fast': {
        'label': 'EGD m=0 fast (lr=0.3, wd=0.1)', 'lr': 0.3, 'wd': 0.1, 'momentum': 0.0,
        'expected_grok': 200,
    },
    'egd_m0_slow': {
        'label': 'EGD m=0 slow (lr=0.01, wd=0.1)', 'lr': 0.01, 'wd': 0.1, 'momentum': 0.0,
        'expected_grok': 4550,
    },
    'egd_m0.9_fast': {
        'label': 'EGD m=0.9 fast (lr=0.1, wd=0.1)', 'lr': 0.1, 'wd': 0.1, 'momentum': 0.9,
        'expected_grok': 450,
    },
    'egd_m0.9_slow': {
        'label': 'EGD m=0.9 slow (lr=0.001, wd=0.1)', 'lr': 0.001, 'wd': 0.1, 'momentum': 0.9,
        'expected_grok': 5200,
    },
}

for name, cfg in CONFIGS.items():
    cfg['phase1_dir'] = SAVE_ROOT / f'{name}'
    cfg['phase2_dir'] = SAVE_ROOT / f'{name}_fixedkf'
    cfg['phase1_dir'].mkdir(parents=True, exist_ok=True)
    cfg['phase2_dir'].mkdir(parents=True, exist_ok=True)
    cfg['specs'] = make_egd_hybrid(cfg['lr'], cfg['wd'], cfg['momentum'])

for name, cfg in CONFIGS.items():
    print(f'  {name:15s} : {cfg["label"]}  (expected eg={cfg["expected_grok"]})')

  egd_m0_fast     : EGD m=0 fast (lr=0.3, wd=0.1)  (expected eg=200)
  egd_m0_slow     : EGD m=0 slow (lr=0.01, wd=0.1)  (expected eg=4550)
  egd_m0.9_fast   : EGD m=0.9 fast (lr=0.1, wd=0.1)  (expected eg=450)
  egd_m0.9_slow   : EGD m=0.9 slow (lr=0.001, wd=0.1)  (expected eg=5200)


# ════════════════════════════════════════════
# PHASE 1 — Training (NO Fourier) + identification
# ════════════════════════════════════════════

## 1.1 — Train 4 configs × 5 seeds (resume-aware)

In [ ]:
for name, cfg in CONFIGS.items():
    print(f'\n═══ Phase 1 — {name} ═══')
    missing = [s for s in SEEDS if not (cfg['phase1_dir'] / f'seed{s}' / 'history.json').exists()]
    done    = [s for s in SEEDS if s not in missing]
    if done:    print(f'  already done : {done}')
    if missing:
        print(f'  will run     : {missing}')
        _ = run_multi_seed(
            BASE_CONFIG, cfg['specs'], seeds=missing,
            label_prefix=f'{name}_p1',
            save_root=str(cfg['phase1_dir']),
            eval_every=EVAL_EVERY,
            fourier_every=None,    # ← PAS de Fourier en Phase 1
            warmup_steps=WARMUP_STEPS,
            verbose_every=5_000, verbose_build=False,
        )
    else:
        print('  all done')


═══ Phase 1 — egd_m0_fast ═══
  will run     : [0, 1, 2, 3, 4]

=== egd_m0_fast_p1 seed 0 (1/5) ===
  [egd_m0_fast_p1_seed0 seed=0] epoch     0 | train acc 0.009 | test acc 0.009


/Users/mverest/Desktop/Fourier/Opti_ML/baseline/optimizer/egd.py:87: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:15.)
  U, S, _ = t.linalg.svd(G, full_matrices=False)  # truncated form


  [egd_m0_fast_p1_seed0] adaptive_logging : grok at epoch 200, switching to eval_every=50, fourier_every=None
  [egd_m0_fast_p1_seed0 seed=0] epoch  5000 | train acc 1.000 | test acc 1.000
  [egd_m0_fast_p1_seed0 seed=0] epoch 10000 | train acc 1.000 | test acc 1.000


## 1.2 — Reload + plot curves (per config)

In [ ]:
for name, cfg in CONFIGS.items():
    runs, _ = load_seeds(cfg['phase1_dir'], SEEDS)
    cfg['runs_1']    = runs
    markers = compute_grokking_markers(runs)
    markers['circuit'] = None   # pas de Fourier en Phase 1 → pas de circuit marker
    cfg['markers_1'] = markers
    cfg['converged'] = [r['seed'] for r in runs if r['history']['test_acc'][-1] >= CONVERGED_THRESH]

    m = markers
    print(f'\n═══ {cfg["label"]} ═══')
    print(f'  mem={m["mem"]}   grok={m["grok"]}   converged={cfg["converged"]}')

    plot_curves_band(
        runs, title=f'Phase 1 — {cfg["label"]}',
        markers=m,
        save_path=FIG_DIR / f'p1_{name}_curves.png',
    )

## 1.3 — Identification per-seed key freqs (sur model.pt final)

In [ ]:
for name, cfg in CONFIGS.items():
    print(f'\n═══ {cfg["label"]} — identification key_freqs ═══')
    converged = cfg['converged']
    if len(converged) < 3:
        print(f'  ⚠ moins de 3 seeds convergés ({converged}) — skip identification'); continue
    per_seed = plot_fourier_components(
        save_root=cfg['phase1_dir'], seeds=converged, config=BASE_CONFIG,
        title=f'Phase 1 — {cfg["label"]}',
        save_path=FIG_DIR / f'p1_{name}_fourier_components.png',
    )
    plot_fourier_components_per_seed(
        save_root=cfg['phase1_dir'], seeds=converged, config=BASE_CONFIG,
        title=f'Phase 1 — {cfg["label"]} — per-seed',
        save_path=FIG_DIR / f'p1_{name}_per_seed.png',
    )
    kf5 = identify_key_freqs(per_seed, k=5)
    cfg['per_seed_freqs'] = kf5['per_seed']
    cfg['key_freqs_majority'] = kf5['majority']
    print(f'  Per-seed top-5 :')
    for s, freqs in kf5['per_seed'].items():
        print(f'    seed {s} : {freqs}')
    print(f'  consensus : {kf5["consensus"]}   majority : {kf5["majority"]}')

## 1.4 — Diagnostic : comparaison des méthodes de sélection key-freqs

**Goal** : pour chaque config × seed, compare 4 méthodes pour identifier les key freqs sur W_L final :

- **topk k=5** : reference Nanda (k fixé, déjà utilisé en Phase 2)
- **cum_energy τ=0.85 / 0.90 / 0.95** : garde les freqs portant τ % de l'énergie de W_L (Parseval)
- **permutation** : test statistique vs null par row-shuffle de W_L (le plus rigoureux, n_perms=500, α=0.05, FDR)

**Conclusion à tirer** : si pour un optimizer le n_keep diffère beaucoup de 5, sa Phase 2 actuelle est sous-optimale → candidat pour re-train dans un nouveau notebook.

⚠ Cette cellule **NE MODIFIE PAS** `cfg['per_seed_freqs']` — la Phase 2 (cellule 2.1) reste inchangée. Les résultats sont stockés dans `cfg['kf_diagnostics']` (nouvelle clé).

In [ ]:
# Inline reload of per-seed Fourier components (no plotting, no existing-code change).
# Then apply 4 selection methods and build a comparison table.

import pandas as pd
from fourier_metrics import make_fourier_basis

_basis    = make_fourier_basis(BASE_CONFIG).cpu()
_cos_rows = _basis[1::2]
_sin_rows = _basis[2::2]

def _load_components(save_root, seeds, p):
    out = {}
    for s in seeds:
        mp = Path(save_root) / f'seed{s}' / 'model.pt'
        if not mp.exists(): continue
        state = t.load(mp, map_location='cpu')
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        def _find(suf):
            for k, v in state.items():
                if k.endswith(suf): return v
            return None
        W_E   = _find('W_E')
        W_U   = _find('W_U')
        W_out = _find('W_out')
        W_E_p = W_E[:, :p].float().cpu()
        W_L   = (W_U[:, :p].float().cpu().T) @ W_out.float().cpu()
        out[s] = {
            'we_cos':     (W_E_p @ _cos_rows.T).norm(dim=0).numpy(),
            'we_sin':     (W_E_p @ _sin_rows.T).norm(dim=0).numpy(),
            'wl_cos':     (_cos_rows @ W_L).norm(dim=1).numpy(),
            'wl_sin':     (_sin_rows @ W_L).norm(dim=1).numpy(),
            '_W_L':       W_L.numpy(),
            '_basis_cos': _cos_rows.numpy(),
            '_basis_sin': _sin_rows.numpy(),
        }
    return out

# ── Apply 4 selection methods per config × seed ────────────────────────────
ALL_ROWS = []
for name, cfg in CONFIGS.items():
    converged = cfg['converged']
    if len(converged) < 3:
        print(f'  ⚠ {name} : moins de 3 seeds convergés — skip'); continue
    comp = _load_components(cfg['phase1_dir'], converged, p=BASE_CONFIG.p)

    kf_topk = identify_key_freqs(comp, method='topk',       k=5,        verbose=False)
    kf_e85  = identify_key_freqs(comp, method='cum_energy', tau=0.85,   verbose=False)
    kf_e90  = identify_key_freqs(comp, method='cum_energy', tau=0.90,   verbose=False)
    kf_e95  = identify_key_freqs(comp, method='cum_energy', tau=0.95,   verbose=False)
    kf_perm = identify_key_freqs(comp, method='permutation',
                                  n_perms=500, alpha=0.05, correction='fdr',
                                  verbose=False)

    cfg['kf_diagnostics'] = {
        'topk_k5':         kf_topk,
        'cum_t0.85':       kf_e85,
        'cum_t0.90':       kf_e90,
        'cum_t0.95':       kf_e95,
        'permutation_fdr': kf_perm,
    }

    for s in sorted(comp.keys()):
        def _fmt(kfd):
            kf = kfd['per_seed'][s]
            return f'({len(kf):2d}) {kf}'
        ALL_ROWS.append({
            'config':         name,
            'seed':           s,
            'topk k=5':       _fmt(kf_topk),
            'cum τ=0.85':     _fmt(kf_e85),
            'cum τ=0.90':     _fmt(kf_e90),
            'cum τ=0.95':     _fmt(kf_e95),
            'permutation':    _fmt(kf_perm),
        })

df_kf = pd.DataFrame(ALL_ROWS)
print('═══ Per-seed key-freq selection — count + freq list ═══')
with pd.option_context('display.max_colwidth', None):
    display(df_kf)

# ── Summary : n_keep mean ± std per config × method ──────────────────────────
summary_rows = []
for name, cfg in CONFIGS.items():
    diag = cfg.get('kf_diagnostics')
    if diag is None: continue
    row = {'config': name}
    for method_name, kf in diag.items():
        counts = [len(v) for v in kf['per_seed'].values()]
        if counts:
            row[method_name] = f'{np.mean(counts):.1f} ± {np.std(counts):.1f}   (range {min(counts)}–{max(counts)})'
    summary_rows.append(row)

print('\n═══ n_keep summary per config × method ═══')
display(pd.DataFrame(summary_rows))

print('\n→ Phase 2 utilise toujours `cfg["per_seed_freqs"]` (topk k=5). Inchangé.')
print('→ Diagnostics dispo dans `cfg["kf_diagnostics"]` pour analyse a posteriori.')

# ════════════════════════════════════════════
# PHASE 2 — Re-train avec key_freqs FIXÉS + Fourier
# ════════════════════════════════════════════

## 2.1 — Re-train per-seed avec `fixed_key_freqs`

In [ ]:
for name, cfg in CONFIGS.items():
    per_seed_freqs = cfg.get('per_seed_freqs', {})
    if not per_seed_freqs:
        print(f'\n⚠ {name} : pas de per_seed_freqs en Phase 1 — skip'); continue
    print(f'\n═══ Phase 2 — {cfg["label"]} — per-seed key_freqs ═══')
    for s in sorted(per_seed_freqs.keys()):
        seed_dir = cfg['phase2_dir'] / f'seed{s}'
        if (seed_dir / 'history.json').exists():
            print(f'  seed {s} ✓ already done — skip'); continue
        seed_kf = per_seed_freqs[s]
        print(f'  → seed {s} : training with key_freqs = {seed_kf}')
        trainer = Trainer(
            BASE_CONFIG, cfg['specs'],
            seed=s, label=f'{name}_p2_seed{s}',
            eval_every=EVAL_EVERY,
            fourier_every=FOURIER_EVERY,
            fixed_key_freqs=seed_kf,
            warmup_steps=WARMUP_STEPS,
            verbose_every=5_000, verbose_build=False,
        )
        trainer.fit()
        trainer.save_run(str(seed_dir))

## 2.2 — Reload + Fourier plots (loss + freq mass + sparsity)

In [ ]:
for name, cfg in CONFIGS.items():
    if not cfg.get('per_seed_freqs'): continue
    seeds_to_reload = sorted(cfg['per_seed_freqs'].keys())
    runs, _ = load_seeds(cfg['phase2_dir'], seeds_to_reload)
    cfg['runs_2']    = runs
    cfg['markers_2'] = compute_grokking_markers(runs)

    print(f'\n═══ {cfg["label"]} — Phase 2 (n={len(runs)} seeds) ═══')
    m = cfg['markers_2']
    print(f'  mem={m["mem"]}   circuit={m["circuit"]}   grok={m["grok"]}')

    plot_fourier_losses_band(
        runs, title=f'Phase 2 — {cfg["label"]} — Fourier loss (per-seed key_freqs)',
        markers=m, save_path=FIG_DIR / f'p2_{name}_fourier_loss.png',
    )
    plot_freq_mass_heatmap(
        runs, title=f'Phase 2 — {cfg["label"]} — frequency masses',
        markers=m, save_path=FIG_DIR / f'p2_{name}_freq_masses.png',
    )
    plot_sparsity(
        runs, title=f'Phase 2 — {cfg["label"]} — sparsity (concentration in key_freqs)',
        markers=m, save_path=FIG_DIR / f'p2_{name}_sparsity.png',
    )

## Summary

In [ ]:
print('═══ EGD SUMMARY ═══\n')
print(f'{"config":18} {"expected eg":>11} {"mem (P1)":>10} {"grok (P1)":>10} {"circuit (P2)":>14} {"grok (P2)":>10}')
print('-' * 100)
for name, cfg in CONFIGS.items():
    m1 = cfg.get('markers_1', {})
    m2 = cfg.get('markers_2', {})
    print(f'{name:18} {cfg["expected_grok"]:>11} '
          f'{str(m1.get("mem")):>10} {str(m1.get("grok")):>10} '
          f'{str(m2.get("circuit")):>14} {str(m2.get("grok")):>10}')
print(f'\nFigures saved to : {FIG_DIR}/')